In [3]:
import pandas as pd
from glob import glob
import ast

Transform a REDFold log to a csv. Also can calculate the F1.

In [4]:
!head -n 10 test.log

current processing 1/66
All_file_list:66
Save RNA test data to data/test.pick
use gpu: True
Loading file:  data/test.pick
Loading dataset Done!!!
train.len=66
FCDenseNet(
  (features): Sequential(
    (0): BasicConv2d(


In [5]:
def log2pd(file_path):
    rows = []
    with open(file_path, "r") as f:
        for line in f:
            if line.startswith(">"):
                label = line[1:-1]
                next_line = f.readline().strip()
                next_next_line = f.readline().strip()
                rows.append(
                    {
                        "id": label,
                        "sequence": next_line,
                        "structure": next_next_line,
                    }
                )
    return pd.DataFrame(rows, columns=["id", "sequence", "structure"])

In [6]:
df = log2pd("test.log")

In [ ]:
# TODO: copy the last F1 score function from sincfold metrics

MATCHING_BRACKETS = [
    ["(", ")"],
    ["[", "]"],
    ["{", "}"],
    ["<", ">"],
    ["A", "a"],
    ["B", "a"],
]

def fold2bp(struc, xop="(", xcl=")"):
    openxs = []
    bps = []
    for i, x in enumerate(struc):
        if x == xop:
            openxs.append(i)
        elif x == xcl:
            if len(openxs) > 0:
                bps.append([openxs.pop() + 1, i + 1])
            else:
                return False
    return bps


def dot2bp(struc):
    bp = []
    for brackets in MATCHING_BRACKETS:
        bp = bp + fold2bp(struc, brackets[0], brackets[1])
    return list(sorted(bp))


def f1_score(ref_bp, pre_bp):
    if len(ref_bp) == 0 and len(pre_bp) == 0:
        return 1
    tp1 = 0
    for rbp in ref_bp:
        # add tolerance of +/- 1 position
        if (
            rbp in pre_bp
            or [rbp[0], rbp[1] - 1] in pre_bp
            or [rbp[0], rbp[1] + 1] in pre_bp
            or [rbp[0] + 1, rbp[1]] in pre_bp
            or [rbp[0] - 1, rbp[1]] in pre_bp
        ):
            tp1 = tp1 + 1
    tp2 = 0
    for pbp in pre_bp:
        if (
            pbp in ref_bp
            or [pbp[0], pbp[1] - 1] in ref_bp
            or [pbp[0], pbp[1] + 1] in ref_bp
            or [pbp[0] + 1, pbp[1]] in ref_bp
            or [pbp[0] - 1, pbp[1]] in ref_bp
        ):
            tp2 = tp2 + 1

    fn = len(ref_bp) - tp1
    fp = len(pre_bp) - tp1

    tpr = pre = f1 = 0.0
    if tp1 + fn > 0:
        tpr = tp1 / float(tp1 + fn)  # sensitivity (=recall =power)
    if tp1 + fp > 0:
        pre = tp2 / float(tp1 + fp)  # precision (=ppv)
    if tpr + pre > 0:
        f1 = 2 * pre * tpr / (pre + tpr)  # F1 score

    return f1

In [8]:
df["base_pairs"] = df["structure"].apply(dot2bp)
df

,id,sequence,structure,base_pairs
0,16s_H.sapiens.mito_domain3,AAGGACCUGGCGGUGCUUCAUAUCCCUCUAGAGGAGCCUGUUCUGU...,..............((((((.............................,"[[15, 307], [16, 306], [17, 305], [18, 304], [..."
1,16s_H.sapiens.mito_domain1,AAUAGGUUUGGUCCUAGCCUUUCUAUUAGCUCUUAGUAAGAUUACA...,...................................((......))....,"[[36, 45], [37, 44], [80, 100], [85, 95], [86,..."
2,16s_D.melanogaster_domain4,ACCGCCCGUCGCUACUACCGAUUGAAUUAUUUAGUGAGGUCUCCGG...,(.((.(......................................)....,"[[1, 50], [3, 49], [4, 48], [6, 45], [60, 74],..."
3,16s_C.reinhardtii.mito_domain2,CCCCCAAGCACGUGCCAGAAGGGUCGGUAAAACGUGCGGUGUCAGU...,.................................................,"[[53, 248], [54, 184], [56, 178], [57, 72], [6..."
4,16s_H.volcanii_domain3,AAGGAAUUGGCGGGGGAGCACUACAACCGGAGGAGCCUGCGGUUUA...,(((.....................(((((.((....)).))))).....,"[[1, 483], [2, 482], [3, 481], [25, 44], [26, ..."
...,...,...,...,...
61,16s_Z.mays_domain2,AUGAUUGGGCGUAAAGCGUCUGUAGGUGGCUUUUCAAGUCCGCCGU...,......................(.((((............)))).....,"[[23, 339], [25, 44], [26, 43], [27, 42], [28,..."
62,16s_H.sapiens.mito_domain4,ACCGCCCGUCACCCUCCUCAAGUAUACUUCAAAGGACAUUUAACUA...,.............((((((..............................,"[[14, 73], [15, 72], [16, 71], [17, 70], [18, ..."
63,16s_G.intestinalis_domain3,AAGGCAUUGACGGAGGGGUACCACCAGACGUGGAGUCUGCGGCUCA...,........(((((.((....................(.(..........,"[[9, 439], [10, 437], [11, 436], [12, 434], [1..."
64,16s_F.ananassa_domain4,ACCGCCCGUCGCUCCUACCGAUUGAAUGGUCCGGUGAAGUUGUUCG...,.................................................,"[[90, 130], [96, 125], [97, 124], [98, 123], [..."


In [9]:
df_ref = pd.read_csv(
    "/home/gkulemeyer/Documents/Repos/RNA-analysis/DataAnalysis/data/sources/ArchiveII.csv",
    index_col="id",
)
df_ref["base_pairs"] = df_ref["base_pairs"].apply(ast.literal_eval)
df_ref

,sequence,structure,base_pairs,len
id,,,,
5s_Acholeplasma-laidlawii-1,UCUGGUGACGAUAGGUAAGAUGGUUCACCUGUUCCCAUCCCGAACA...,((((((((......((((((((....((((((.............)...,"[[1, 111], [2, 110], [3, 109], [4, 108], [5, 1...",112
5s_Acidovorax-temperans-1,UGCCUGAUGACCAUAGCAAGUUGGUACCACUCCUUCCCAUCCCGAA...,.(((((((((.....((((((((.....((((((...............,"[[2, 115], [3, 114], [4, 113], [5, 112], [6, 1...",115
tmRNA_Stre.gord._TRW-29390_1-349,GGGGUCGUUACGGAUUCGACAGGCAUUAUGAGGCAUAUUUUGCGAC...,(((((((............((((((((....(((((((((..((((...,"[[1, 345], [2, 344], [3, 343], [4, 342], [5, 3...",349
tRNA_tdbR00000055-Schizosaccharomyces_pombe-4896-Glu-3UC,UCCGUUGUGGUCCAACGGCUAGGAUUCGUCGCUUUCACCGACGGGA...,(((((((..((((........))))((((((.......)))))).....,"[[1, 71], [2, 70], [3, 69], [4, 68], [5, 67], ...",75
srp_List.mono._U15684,UGGGUUGAUGAGCGUGAAGCCUUCGCUCGGUUGGAUUUUUCUUCAU...,.(.((((...(.(.((.(.((..(.....)..)).)...(...(.....,"[[2, 276], [4, 274], [5, 273], [6, 272], [7, 2...",279
...,...,...,...,...
5s_Bacillus-cereus-6,UGGUAAUGAUGGCAGAGAGGUCACACCCGUUCCCAUACCGAACACG...,((((((.....((((((((.....((((((.............)))...,"[[1, 111], [2, 110], [3, 109], [4, 108], [5, 1...",114
srp_Myco.aviu._AE016958,GGGGACCCCGCGCACCCGACAGAGCCCGUUGACCCUUGCUGCCUUC...,((((.....(.(...(.(.....(.(....).).....).).).)....,"[[1, 53], [2, 52], [3, 51], [4, 50], [10, 45],...",88
tmRNA_Heli.pylo._AE001503_1-383,GGGGCUGACUUGGAUUUCGACAGAUUUCUUGUCGCACAGAUAGCAU...,(((((((.............((((..((((((((.((((((...((...,"[[1, 382], [2, 381], [3, 380], [4, 379], [5, 3...",383


In [10]:
print(df_ref.loc[df.id, "base_pairs"])
# print(df_ref.loc["16s_H.sapiens.mito_domain3", "base_pairs"])

id
16s_H.sapiens.mito_domain3        [[8, 314], [9, 313], [10, 311], [11, 310], [12...
16s_H.sapiens.mito_domain1        [[4, 20], [5, 19], [6, 18], [7, 17], [8, 16], ...
16s_D.melanogaster_domain4        [[2, 136], [4, 133], [7, 129], [8, 128], [10, ...
16s_C.reinhardtii.mito_domain2    [[8, 37], [9, 36], [10, 35], [11, 34], [12, 33...
16s_H.volcanii_domain3            [[8, 488], [9, 487], [10, 485], [11, 484], [12...
                                                        ...                        
16s_Z.mays_domain2                [[8, 325], [9, 324], [10, 323], [11, 308], [12...
16s_H.sapiens.mito_domain4        [[4, 85], [7, 81], [8, 80], [10, 78], [14, 73]...
16s_G.intestinalis_domain3        [[8, 440], [9, 439], [10, 437], [11, 436], [12...
16s_F.ananassa_domain4            [[2, 139], [4, 136], [7, 132], [8, 131], [10, ...
16s_C.elegans_domain1             [[4, 20], [5, 19], [6, 18], [7, 17], [8, 16], ...
Name: base_pairs, Length: 66, dtype: object


In [11]:
df["base_pairs"] = df["structure"].apply(dot2bp)

df["test_f1"] = df.apply(
    lambda row: f1_score(
        df_ref.loc[row.id, "base_pairs"],  # lista de pares ref
        row["base_pairs"],  # lista de pares pred
    ),
    axis=1,
)
df.test_f1.mean()

np.float64(0.35292142812124117)

In [12]:
# # all partitions kfold
# log_files = sorted(glob("results/kfold230423/test*.log"))

# pred_pds = []
# for logf in log_files:
#     print(logf)
#     dfx = log2pd(logf)
#     dfx["base_pairs"] = dfx["structure"].apply(dot2bp)
#     dfx.to_csv(logf.replace(".log", ".csv"), index=False)